# ChromoDiff-Absorb: Generative Zero-Shot Pathogenicity Prediction via Discrete Genomic Diffusion

This notebook implements **ChromoDiff** (formalized as ChromoDiff-Absorb), a categorical 1D dilated residual diffusion model that learns the non-coding syntax of the human genome. It contains:
1. **Automatic Project Builder**: Writes out all the modular Python files (`src/` and `configs/`) so that the project is completely self-contained and modular on your Kaggle instance.
2. **Data Preprocessor**: Downloads `clinvar.vcf` and `hg38.fa`, filters SNPs, and tokenizes sequence windows (or generates high-quality synthetic data for fast dry-runs).
3. **Model Training**: Denoises discrete sequences by learning to reconstruct masked bases with a selective cross-entropy loss.
4. **Zero-Shot GVES Scoring**: Computes the Generative Variant Effect Score (GVES) at evaluation time to predict ClinVar pathogenicity.

## Setup & GPU Check

In [ ]:
# 1. Verify GPU acceleration availability on Kaggle
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name  :", torch.cuda.get_device_name(0))

# 2. Install required package dependency for Fasta reading
!pip install -q pyfaidx pandas numpy tqdm pyyaml matplotlib scikit-learn seaborn

## Step 1: Write Modular Project Code

Run the cell below to write out the package directories and files.

In [ ]:
import os

# Create directory structures
os.makedirs("configs", exist_ok=True)
os.makedirs("src/models", exist_ok=True)

# Write base_config.yaml
with open("configs/base_config.yaml", "w") as f:
    f.write("""# Diffusion Schedule Settings
T_STEPS: 1000
BETA_START: 0.0001
BETA_END: 0.02
VOCAB_SIZE: 6                # A=0, C=1, G=2, T=3, N=4, [MASK]=5
SEQ_LEN: 1024
MIN_CORRUPTION_RATE: 0.15    # Floor for token masking during diffusion

# Model Architecture
HIDDEN_DIM: 256

# Training Hyperparameters
BATCH_SIZE: 64
EPOCHS: 50
LEARNING_RATE: 0.001         # Base learning rate
GRAD_CLIP: 1.0
WARMUP_EPOCHS: 2             # Linear warmup duration (was 5 - too long)
T_0: 25                      # CosineAnnealingWarmRestarts restart cycle
ETA_MIN: 0.000001            # Floor for Cosine learning rate

# Paths & Directories
DATA_DIR: "data/processed"
CHECKPOINT_DIR: "outputs/checkpoints"
SEED: 42
""")

# Write src/utils.py
with open("src/utils.py", "w") as f:
    f.write("""import os
import yaml
import torch
import random
import logging
import numpy as np

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_config(config_path: str) -> dict:
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Configuration file not found: {config_path}")
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def setup_logger(name: str = "ChromoDiff") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        ch = logging.StreamHandler()
        ch.setLevel(logging.INFO)
        formatter = logging.Formatter(
            "[%(asctime)s][%(name)s][%(levelname)s] %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S"
        )
        ch.setFormatter(formatter)
        logger.addHandler(ch)
    return logger

def setup_dirs(*dirs):
    for d in dirs:
        if d:
            os.makedirs(d, exist_ok=True)
""")

# Write src/dataset.py
with open("src/dataset.py", "w") as f:
    f.write("""import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class GenomicDataset(Dataset):
    def __init__(self, data: torch.Tensor):
        if not isinstance(data, torch.Tensor):
            self.data = torch.tensor(data, dtype=torch.long)
        else:
            self.data = data.long()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def reverse_complement_tokens(x: np.ndarray) -> np.ndarray:
    comp_map = np.array([3, 2, 1, 0, 4, 5], dtype=np.int8)
    return comp_map[x[:, ::-1]]

def reverse_complement_tensor(x: torch.Tensor) -> torch.Tensor:
    comp_map = torch.tensor([3, 2, 1, 0, 4, 5], dtype=torch.long, device=x.device)
    reversed_x = torch.flip(x, dims=[-1])
    return comp_map[reversed_x]

def get_dataloader(data_path: str, batch_size: int, shuffle: bool = True, num_workers: int = 0) -> DataLoader:
    data_np = np.load(data_path)
    data_tensor = torch.tensor(data_np, dtype=torch.long)
    dataset = GenomicDataset(data_tensor)
    
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
        num_workers=num_workers
    )
    return loader
""")

# Write src/diffusion.py
with open("src/diffusion.py", "w") as f:
    f.write("""import torch

class AbsorbingStateScheduler:
    def __init__(self, num_steps: int = 1000, beta_start: float = 1e-4, beta_end: float = 0.02, min_corruption_rate: float = 0.15):
        self.num_steps = num_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.min_corruption_rate = min_corruption_rate

        self.betas = torch.linspace(beta_start, beta_end, num_steps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

    def to(self, device: torch.device):
        self.betas = self.betas.to(device)
        self.alphas = self.alphas.to(device)
        self.alphas_cumprod = self.alphas_cumprod.to(device)
        return self

    def sample_timesteps(self, batch_size: int, device: torch.device) -> torch.Tensor:
        u = torch.rand(batch_size, device=device)
        t = (u ** 2 * self.num_steps).long().clamp(0, self.num_steps - 1)
        return t

    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, L = x_start.shape
        device = x_start.device
        
        if self.alphas_cumprod.device != device:
            self.alphas_cumprod = self.alphas_cumprod.to(device)

        a_bar = self.alphas_cumprod[t].unsqueeze(1) # [B, 1]
        a_bar_floored = torch.clamp(a_bar, max=1.0 - self.min_corruption_rate)

        rand_probs = torch.rand((B, L), device=device)
        mutate_mask = rand_probs > a_bar_floored # [B, L] Bool

        x_noisy = torch.where(mutate_mask, torch.tensor(5, device=device, dtype=torch.long), x_start)
        return x_noisy, mutate_mask
""")

# Write src/models/embedding.py
with open("src/models/embedding.py", "w") as f:
    f.write("""import math
import torch
import torch.nn as nn

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, time: torch.Tensor) -> torch.Tensor:
        device = time.device
        half_dim = self.dim // 2
        freq = math.log(10000) / (half_dim - 1)
        freq = torch.exp(torch.arange(half_dim, device=device) * -freq)
        angles = time[:, None].float() * freq[None, :]
        return torch.cat([angles.sin(), angles.cos()], dim=-1)
""")

# Write src/models/unet.py
with open("src/models/unet.py", "w") as f:
    f.write("""import torch
import torch.nn as nn
from .embedding import SinusoidalPositionEmbeddings

class DilatedResidualBlock(nn.Module):
    def __init__(self, hidden_dim: int, dilation: int):
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, hidden_dim)
        self.act1 = nn.GELU()
        
        self.time_proj = nn.Linear(hidden_dim, hidden_dim)
        
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation)
        self.norm2 = nn.GroupNorm(8, hidden_dim)
        
    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x)
        h = self.act1(self.norm1(h))
        t_proj = self.time_proj(t_emb).unsqueeze(2)
        h = h + t_proj
        h = self.act1(self.norm2(self.conv2(h)))
        return x + h

class GenoDiff1D(nn.Module):
    def __init__(self, vocab_size: int = 6, hidden_dim: int = 256, dilations: list = None):
        super().__init__()
        if dilations is None:
            dilations = [1, 2, 4, 8, 16, 32]

        self.dna_embedding = nn.Embedding(vocab_size, hidden_dim)

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )

        self.res_blocks = nn.ModuleList([
            DilatedResidualBlock(hidden_dim, dilation=d) for d in dilations
        ])

        self.output_norm = nn.GroupNorm(8, hidden_dim)
        self.final_conv = nn.Conv1d(hidden_dim, vocab_size, kernel_size=1)

    def forward(self, noisy_dna: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        x = self.dna_embedding(noisy_dna).permute(0, 2, 1)
        t_emb = self.time_mlp(t)

        for block in self.res_blocks:
            x = block(x, t_emb)

        logits = self.final_conv(self.output_norm(x))
        return logits
""")

# Write src/models/__init__.py
with open("src/models/__init__.py", "w") as f:
    f.write("""from .embedding import SinusoidalPositionEmbeddings
from .unet import DilatedResidualBlock, GenoDiff1D
""")

# Write src/train.py
with open("src/train.py", "w") as f:
    f.write("""import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm

from .utils import set_seed, load_config, setup_logger, setup_dirs
from .dataset import get_dataloader, reverse_complement_tensor
from .diffusion import AbsorbingStateScheduler
from .models.unet import GenoDiff1D

def get_lr(optimizer):
    return optimizer.param_groups[0][\"lr\"]

def linear_warmup(step, warmup_steps, base_lr):
    return base_lr * min(1.0, step / max(warmup_steps, 1))

def train_model(config_path: str):
    config = load_config(config_path)
    set_seed(config.get(\"SEED\", 42))
    logger = setup_logger(\"ChromoDiff.Train\")
    
    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")
    logger.info(f\"Using device: {device}\")
    
    checkpoint_dir = config.get(\"CHECKPOINT_DIR\", \"outputs/checkpoints\")
    data_dir = config.get(\"DATA_DIR\", \"data/processed\")
    setup_dirs(checkpoint_dir)
    
    train_data_path = os.path.join(data_dir, \"X_healthy.npy\")
    if not os.path.exists(train_data_path):
        logger.error(f\"Training data not found at {train_data_path}. Please run preprocessing first.\")
        return
        
    logger.info(f\"Loading data from {train_data_path}...\")
    train_loader = get_dataloader(
        data_path=train_data_path,
        batch_size=config[\"BATCH_SIZE\"],
        shuffle=True,
        num_workers=0
    )
    
    vocab_size = config.get(\"VOCAB_SIZE\", 6)
    hidden_dim = config.get(\"HIDDEN_DIM\", 256)
    model = GenoDiff1D(vocab_size=vocab_size, hidden_dim=hidden_dim).to(device)
    
    num_steps = config.get(\"T_STEPS\", 1000)
    scheduler_diffusion = AbsorbingStateScheduler(
        num_steps=num_steps,
        beta_start=config.get(\"BETA_START\", 1e-4),
        beta_end=config.get(\"BETA_END\", 0.02),
        min_corruption_rate=config.get(\"MIN_CORRUPTION_RATE\", 0.15)
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=config[\"LEARNING_RATE\"], weight_decay=1e-4)
    
    lr_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=config.get(\"T_0\", 25),
        T_mult=1,
        eta_min=config.get(\"ETA_MIN\", 1e-6)
    )
    
    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler(\"cuda\", enabled=use_amp)
    
    best_loss = float(\"inf\")
    train_history = []
    
    warmup_epochs = config.get(\"WARMUP_EPOCHS\", 2)
    warmup_steps = warmup_epochs * len(train_loader)
    global_step = 0
    epochs = config.get(\"EPOCHS\", 50)
    
    logger.info(\"Starting Unsupervised Diffusion Training...\")
    logger.info(f\"  Model params : {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M\")
    logger.info(f\"  LR warmup    : {warmup_epochs} epochs ({warmup_steps} steps)\")
    logger.info(f\"  LR restarts  : every {config.get('T_0', 25)} epochs\")
    logger.info(f\"  RC augment   : 50% per batch\")
    logger.info(f\"  Timestep samp: importance-weighted (squared)\")
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f\"Epoch {epoch:02d}/{epochs:02d}\")
        for batch_idx, x_start in enumerate(progress_bar):
            if global_step < warmup_steps:
                warm_lr = linear_warmup(global_step, warmup_steps, config[\"LEARNING_RATE\"])
                for pg in optimizer.param_groups:
                    pg[\"lr\"] = warm_lr
                    
            x_start = x_start.to(device, non_blocking=True)
            
            if torch.rand(1).item() < 0.5:
                x_start = reverse_complement_tensor(x_start)
            
            t = scheduler_diffusion.sample_timesteps(x_start.shape[0], device)
            x_noisy, mutate_mask = scheduler_diffusion.q_sample(x_start, t)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast(\"cuda\", enabled=use_amp):
                predicted_logits = model(x_noisy, t)
                
                mask_flat = mutate_mask.view(-1)
                logits_flat = predicted_logits.permute(0, 2, 1).reshape(-1, vocab_size)
                labels_flat = x_start.view(-1)
                
                logits_masked = logits_flat[mask_flat]
                labels_masked = labels_flat[mask_flat]
                
                if mask_flat.sum() > 0:
                    loss = F.cross_entropy(logits_masked, labels_masked)
                else:
                    loss = F.cross_entropy(logits_flat, labels_flat)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.get(\"GRAD_CLIP\", 1.0))
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            global_step += 1
            
            progress_bar.set_postfix({
                \"loss\": f\"{loss.item():.4f}\",
                \"lr\": f\"{get_lr(optimizer):.2e}\"
            })
            
        if global_step >= warmup_steps:
            lr_scheduler.step(epoch - warmup_epochs + 1)
            
        avg_loss = epoch_loss / len(train_loader)
        train_history.append(avg_loss)
        
        logger.info(f\"Epoch {epoch:02d} | Avg Loss: {avg_loss:.4f} | LR: {get_lr(optimizer):.2e}\")
        
        ckpt = {
            \"epoch\": epoch,
            \"model_state_dict\": model.state_dict(),
            \"optimizer_state\": optimizer.state_dict(),
            \"avg_loss\": avg_loss,
        }
        torch.save(ckpt, os.path.join(checkpoint_dir, f\"genodiff_epoch_{epoch:03d}.pth\"))
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), os.path.join(checkpoint_dir, \"genodiff_best.pth\"))
            logger.info(f\"  New best model saved with Loss: {best_loss:.4f}\")
            
    logger.info(f\"Training completed successfully! Best loss: {best_loss:.4f}\")
    
    plt.figure(figsize=(10, 4))
    plt.plot(train_history, lw=2, color=\"steelblue\", label=\"Avg Masked Cross Entropy Loss\")
    plt.xlabel(\"Epoch\")
    plt.ylabel(\"Loss\")
    plt.title(\"ChromoDiff Training Loss Curve\")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(checkpoint_dir, \"training_curve.png\"), dpi=150)
    plt.close()
    logger.info(f\"Saved loss curve to {os.path.join(checkpoint_dir, 'training_curve.png')}\")
""")

# Write src/evaluate.py
with open("src/evaluate.py", "w") as f:
    f.write("""import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

from .utils import load_config, setup_logger, setup_dirs
from .models.unet import GenoDiff1D

def calculate_gves(model, seq_corrupted, ref_base, alt_base, mutation_pos=512, epsilon=1e-8):
    model.eval()
    device = next(model.parameters()).device
    
    nuc_to_idx = {\"A\": 0, \"C\": 1, \"G\": 2, \"T\": 3, \"N\": 4}
    ref_idx = nuc_to_idx[ref_base] if isinstance(ref_base, str) else ref_base
    alt_idx = nuc_to_idx[alt_base] if isinstance(alt_base, str) else alt_base
    
    if seq_corrupted.dim() == 1:
        seq_corrupted = seq_corrupted.unsqueeze(0)
    
    seq_corrupted = seq_corrupted.to(device)
    B = seq_corrupted.shape[0]
    
    seq_masked = seq_corrupted.clone()
    seq_masked[:, mutation_pos] = 5
    
    t_tensor = torch.zeros(B, device=device, dtype=torch.long)
    
    with torch.no_grad():
        with torch.amp.autocast(\"cuda\", enabled=torch.cuda.is_available()):
            logits = model(seq_masked, t_tensor)
        
        probs = torch.softmax(logits[:, :, mutation_pos], dim=1)
        p_ref = probs[:, ref_idx]
        p_alt = probs[:, alt_idx]
        gves = torch.log(p_ref + epsilon) - torch.log(p_alt + epsilon)
        
    return gves.cpu().numpy()

def score_dataset_gves(model, X_healthy, X_corrupted, mutation_pos=512, batch_size=64, epsilon=1e-8):
    model.eval()
    device = next(model.parameters()).device
    N = len(X_healthy)
    all_gves = []
    
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch_h = X_healthy[start:end].to(device)
        batch_c = X_corrupted[start:end].to(device)
        B = batch_h.shape[0]
        
        batch_masked = batch_c.clone()
        batch_masked[:, mutation_pos] = 5
        t_tensor = torch.zeros(B, device=device, dtype=torch.long)
        
        with torch.no_grad():
            with torch.amp.autocast(\"cuda\", enabled=torch.cuda.is_available()):
                logits = model(batch_masked, t_tensor)
            probs = torch.softmax(logits[:, :, mutation_pos], dim=1)
            
            ref_idx = batch_h[:, mutation_pos]
            alt_idx = batch_c[:, mutation_pos]
            
            p_ref = probs[torch.arange(B), ref_idx]
            p_alt = probs[torch.arange(B), alt_idx]
            gves = torch.log(p_ref + epsilon) - torch.log(p_alt + epsilon)
            
        all_gves.append(gves.cpu().numpy())
        
    return np.concatenate(all_gves)

def evaluate_predictions(y_true, scores, best_name=\"GVES\", checkpoint_dir=\"outputs/checkpoints\"):
    auroc = roc_auc_score(y_true, scores)
    auprc = average_precision_score(y_true, scores)
    
    auroc_flip = roc_auc_score(y_true, -scores)
    if auroc_flip > auroc:
        scores = -scores
        auroc = auroc_flip
        auprc = average_precision_score(y_true, scores)
        flip_note = \" (flipped)\"
    else:
        flip_note = \"\"
        
    fpr, tpr, _ = roc_curve(y_true, scores)
    prec, rec, _ = precision_recall_curve(y_true, scores)
    rand_auprc = y_true.mean()
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f\"ChromoDiff Evaluation Metrics — {best_name} Scoring{flip_note}\", fontsize=13, fontweight=\"bold\")
    
    axes[0].plot(fpr, tpr, color=\"steelblue\", lw=2, label=f\"AUROC={auroc:.4f}\")
    axes[0].plot([0, 1], [0, 1], \"k--\", lw=1, alpha=0.4, label=\"Random\")
    axes[0].fill_between(fpr, tpr, alpha=0.1, color=\"steelblue\")
    axes[0].set(xlabel=\"FPR\", ylabel=\"TPR\", title=\"ROC Curve\")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(rec, prec, color=\"coral\", lw=2, label=f\"AUPRC={auprc:.4f}\")
    axes[1].axhline(rand_auprc, color=\"k\", ls=\"--\", lw=1, alpha=0.4, label=f\"Random={rand_auprc:.3f}\")
    axes[1].fill_between(rec, prec, alpha=0.1, color=\"coral\")
    axes[1].set(xlabel=\"Recall\", ylabel=\"Precision\", title=\"PR Curve\")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].hist(scores[y_true == 0], bins=60, alpha=0.6, color=\"steelblue\", label=\"Benign\", density=True)
    axes[2].hist(scores[y_true == 1], bins=60, alpha=0.6, color=\"coral\", label=\"Pathogenic\", density=True)
    axes[2].set(xlabel=\"Score\", ylabel=\"Density\", title=\"Score Distribution\")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(checkpoint_dir, \"evaluation_metrics.png\")
    plt.savefig(plot_path, dpi=150)
    plt.close()
    
    return auroc, auprc, plot_path

def run_evaluation(config_path: str, weights_path: str):
    config = load_config(config_path)
    logger = setup_logger(\"ChromoDiff.Evaluate\")
    
    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")
    logger.info(f\"Using device: {device}\")
    
    checkpoint_dir = config.get(\"CHECKPOINT_DIR\", \"outputs/checkpoints\")
    data_dir = config.get(\"DATA_DIR\", \"data/processed\")
    setup_dirs(checkpoint_dir)
    
    logger.info(\"Loading validation datasets...\")
    healthy_path = os.path.join(data_dir, \"X_healthy.npy\")
    corrupted_path = os.path.join(data_dir, \"X_corrupted.npy\")
    labels_path = os.path.join(data_dir, \"Y_labels.npy\")
    
    if not (os.path.exists(healthy_path) and os.path.exists(corrupted_path) and os.path.exists(labels_path)):
        logger.error(\"Dataset arrays not found. Please run preprocessing first.\")
        return
        
    X_healthy = torch.tensor(np.load(healthy_path), dtype=torch.long)
    X_corrupted = torch.tensor(np.load(corrupted_path), dtype=torch.long)
    Y_labels = np.load(labels_path)
    
    vocab_size = config.get(\"VOCAB_SIZE\", 6)
    hidden_dim = config.get(\"HIDDEN_DIM\", 256)
    model = GenoDiff1D(vocab_size=vocab_size, hidden_dim=hidden_dim).to(device)
    
    logger.info(f\"Loading pretrained weights from {weights_path}...\")
    state_dict = torch.load(weights_path, map_location=device)
    if \"model_state_dict\" in state_dict:
        model.load_state_dict(state_dict[\"model_state_dict\"])
    else:
        model.load_state_dict(state_dict)
        
    logger.info(\"Computing zero-shot GVES pathogenicity scores...\")
    gves_scores = score_dataset_gves(
        model=model,
        X_healthy=X_healthy,
        X_corrupted=X_corrupted,
        mutation_pos=512,
        batch_size=config.get(\"BATCH_SIZE\", 64)
    )
    
    auroc, auprc, plot_path = evaluate_predictions(
        y_true=Y_labels,
        scores=gves_scores,
        best_name=\"GVES\",
        checkpoint_dir=checkpoint_dir
    )
    
    logger.info(\"==================================================\")
    logger.info(\"  Zero-Shot Variant Pathogenicity Metrics\")
    logger.info(\"==================================================\")
    logger.info(f\"  AUROC : {auroc:.4f}\")
    logger.info(f\"  AUPRC : {auprc:.4f}\")
    logger.info(f\"  Saved evaluation figures → {plot_path}\")
    logger.info(\"==================================================\")
""")

# Write src/__init__.py
with open("src/__init__.py", "w") as f:
    f.write("""# Package init
""")

# Write src/preprocess.py
with open("src/preprocess.py", "w") as f:
    f.write("""import os
import urllib.request
import gzip
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

from .utils import load_config, setup_logger, setup_dirs

NUC_TO_IDX = {\"A\": 0, \"C\": 1, \"G\": 2, \"T\": 3, \"N\": 4}
IDX_TO_NUC = {v: k for k, v in NUC_TO_IDX.items()}

BYTE_LUT = np.full(256, 4, dtype=np.int8)
for base, idx in NUC_TO_IDX.items():
    BYTE_LUT[ord(base)] = idx

def seq_to_tokens(seq: str) -> np.ndarray:
    arr = np.frombuffer(seq.encode(\"ascii\"), dtype=np.uint8)
    return BYTE_LUT[arr]

def download_file(url: str, dest_path: str, logger):
    if os.path.exists(dest_path):
        logger.info(f\"File {dest_path} already exists. Skipping download.\")
        return
    logger.info(f\"Downloading {url} to {dest_path}...\")
    class TqdmUpTo(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)
    with TqdmUpTo(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(dest_path)) as t:
        urllib.request.urlretrieve(url, filename=dest_path, reporthook=t.update_to)

def extract_gzip(src_path: str, dest_path: str, logger):
    if os.path.exists(dest_path):
        logger.info(f\"Extracted file {dest_path} already exists. Skipping extraction.\")
        return
    logger.info(f\"Extracting {src_path} to {dest_path}...\")
    with gzip.open(src_path, 'rb') as f_in:
        with open(dest_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    logger.info(\"Extraction complete.\")

def generate_dummy_data(dest_dir: str, num_healthy: int = 2000, num_variant: int = 1000, seq_len: int = 1024, seed: int = 42):
    np.random.seed(seed)
    setup_dirs(dest_dir)
    probs = [0.24, 0.24, 0.24, 0.24, 0.04]
    X_healthy = np.random.choice(5, size=(num_healthy, seq_len), p=probs).astype(np.int8)
    X_eval_ref = np.random.choice(4, size=(num_variant, seq_len)).astype(np.int8)
    X_eval_alt = X_eval_ref.copy()
    mutation_pos = seq_len // 2
    for i in range(num_variant):
        ref_base = X_eval_ref[i, mutation_pos]
        choices = [b for b in range(4) if b != ref_base]
        X_eval_alt[i, mutation_pos] = np.random.choice(choices)
    Y_labels = np.random.choice([0, 1], size=(num_variant,)).astype(np.int8)
    np.save(os.path.join(dest_dir, \"X_healthy.npy\"), X_eval_ref)
    np.save(os.path.join(dest_dir, \"X_corrupted.npy\"), X_eval_alt)
    np.save(os.path.join(dest_dir, \"Y_labels.npy\"), Y_labels)

def preprocess_pipeline(config_path: str, dummy: bool = False):
    config = load_config(config_path)
    logger = setup_logger(\"ChromoDiff.Preprocess\")
    data_dir = config.get(\"DATA_DIR\", \"data/processed\")
    raw_dir = \"data/raw\"
    setup_dirs(data_dir, raw_dir)
    
    if dummy:
        logger.info(\"Generating synthetic dummy data for testing pipeline...\")
        generate_dummy_data(data_dir, seed=config.get(\"SEED\", 42))
        logger.info(f\"Synthetic data saved successfully to {data_dir}!\")
        return
        
    logger.info(\"Starting raw genomic data download and extraction...\")
    clinvar_url = \"https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz\"
    clinvar_gz = os.path.join(raw_dir, \"clinvar.vcf.gz\")
    clinvar_vcf = os.path.join(raw_dir, \"clinvar.vcf\")
    try:
        download_file(clinvar_url, clinvar_gz, logger)
        extract_gzip(clinvar_gz, clinvar_vcf, logger)
    except Exception as e:
        logger.error(f\"Failed to download/extract ClinVar: {e}\")
        logger.info(\"Falling back to dummy mode.\")
        generate_dummy_data(data_dir, seed=config.get(\"SEED\", 42))
        return
        
    hg38_url = \"https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz\"
    hg38_gz = os.path.join(raw_dir, \"hg38.fa.gz\")
    hg38_fa = os.path.join(raw_dir, \"hg38.fa\")
    try:
        download_file(hg38_url, hg38_gz, logger)
        extract_gzip(hg38_gz, hg38_fa, logger)
    except Exception as e:
        logger.error(f\"Failed to download/extract hg38: {e}\")
        logger.info(\"Falling back to dummy mode.\")
        generate_dummy_data(data_dir, seed=config.get(\"SEED\", 42))
        return
        
    try:
        from pyfaidx import Fasta
    except ImportError:
        logger.error(\"pyfaidx library is missing. Install requirements first.\")
        return
        
    logger.info(\"Parsing files and generating token dataset...\")
    try:
        genome = Fasta(hg38_fa, as_raw=True, sequence_always_upper=True)
    except Exception as e:
        logger.error(f\"Error opening hg38.fa using pyfaidx: {e}\")
        logger.info(\"Falling back to dummy data generation.\")
        generate_dummy_data(data_dir, seed=config.get(\"SEED\", 42))
        return
        
    logger.info(\"Parsing ClinVar SNPs...\")
    allowed_chroms = [str(i) for i in range(1, 23)] + [\"X\", \"Y\"]
    rows = []
    with open(clinvar_vcf, \"r\") as f:
        for line in tqdm(f, desc=\"Parsing VCF\"):
            if line.startswith(\"#\"):
                continue
            parts = line.rstrip(\"\\n\").split(\"\\t\")
            chrom = parts[0]
            pos = int(parts[1])
            ref = parts[3]
            alt = parts[4]
            info = parts[7]
            if chrom not in allowed_chroms:
                continue
            if len(ref) != 1:
                continue
            if \",\" in alt or len(alt) != 1:
                continue
            if ref not in NUC_TO_IDX or alt not in NUC_TO_IDX:
                continue
            if \"CLNSIG=Pathogenic\" in info or \"CLNSIG=Likely_pathogenic\" in info:
                label = 1
            elif \"CLNSIG=Benign\" in info or \"CLNSIG=Likely_benign\" in info:
                label = 0
            else:
                continue
            rows.append((f\"chr{chrom}\", pos, ref, alt, label))
            
    df = pd.DataFrame(rows, columns=[\"chrom\", \"pos\", \"ref\", \"alt\", \"label\"])
    logger.info(f\"Found {len(df)} eligible SNPs.\")
    
    WINDOW_SIZE = 1024
    MUT_POS = 512
    MAX_N_FRAC = 0.02
    
    ref_windows = []
    alt_windows = []
    labels = []
    
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=\"Extracting windows\"):
        chrom = row.chrom
        if chrom not in genome.keys():
            continue
        pos1 = int(row.pos)
        start0 = pos1 - 1 - MUT_POS
        end0 = start0 + WINDOW_SIZE
        if start0 < 0:
            continue
        seq = genome[chrom][start0:end0]
        if len(seq) != WINDOW_SIZE:
            continue
        ref_tokens = seq_to_tokens(seq)
        r_idx = NUC_TO_IDX[row.ref]
        a_idx = NUC_TO_IDX[row.alt]
        if ref_tokens[MUT_POS] != r_idx:
            continue
        if (ref_tokens == 4).mean() > MAX_N_FRAC:
            continue
        alt_tokens = ref_tokens.copy()
        alt_tokens[MUT_POS] = a_idx
        
        ref_windows.append(ref_tokens)
        alt_windows.append(alt_tokens)
        labels.append(int(row.label))
        
    X_eval_ref = np.asarray(ref_windows, dtype=np.int8)
    X_eval_alt = np.asarray(alt_windows, dtype=np.int8)
    Y_labels = np.asarray(labels, dtype=np.int8)
    
    np.save(os.path.join(data_dir, \"X_healthy.npy\"), X_eval_ref)
    np.save(os.path.join(data_dir, \"X_corrupted.npy\"), X_eval_alt)
    np.save(os.path.join(data_dir, \"Y_labels.npy\"), Y_labels)
    logger.info(f\"Datasets generated successfully! Saved to {data_dir}.\")
""")
print("Modular packages created successfully! Directory structure:")
print("  - configs/base_config.yaml")
print("  - src/__init__.py")
print("  - src/utils.py")
print("  - src/dataset.py")
print("  - src/diffusion.py")
print("  - src/preprocess.py")
print("  - src/train.py")
print("  - src/evaluate.py")
print("  - src/models/__init__.py")
print("  - src/models/embedding.py")
print("  - src/models/unet.py")

## Step 2: Data Preprocessing

Generate/download the dataset. Set `USE_DUMMY_DATA = False` to run on the full ClinVar dataset on Kaggle.

In [ ]:
# Config options
USE_DUMMY_DATA = False   # Set to True for a fast 1-minute test; False to download full hg38 and ClinVar VCF

from src.preprocess import preprocess_pipeline
preprocess_pipeline("configs/base_config.yaml", dummy=USE_DUMMY_DATA)

## Step 3: Model Training

Trains the discrete absorbing diffusion model.

In [ ]:
from src.train import train_model
train_model("configs/base_config.yaml")

## Step 4: Zero-Shot GVES Pathogenicity Evaluation

Evaluates the trained model zero-shot on variants centered at position 512 using GVES:
$$\text{GVES} = \log(P_{ref} + \epsilon) - \log(P_{alt} + \epsilon)$$
It will print the final zero-shot AUROC and AUPRC metrics and output the evaluation charts.

In [ ]:
from src.evaluate import run_evaluation
run_evaluation("configs/base_config.yaml", "outputs/checkpoints/genodiff_best.pth")

## Step 5: Visualize Metrics

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

metrics_img_path = "outputs/checkpoints/evaluation_metrics.png"
if os.path.exists(metrics_img_path):
    img = Image.open(metrics_img_path)
    plt.figure(figsize=(12, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
else:
    print("Metrics plot not found. Make sure Step 4 completed successfully.")